# Experiment of Algorithm Selection Varying Reward Noise

In this notebook, we compare the selection accuracy of AVG and MID using A/B testing data (and IPS using offline data) varying the reward noise.

In [1]:
import os
import contextlib
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from scipy import stats

from dataset import KuaiRecDataset
from estimation import run_experiment
from utils import evaluate_performance

warnings.filterwarnings('ignore')

## Experiment

In [2]:
# experiment configuration
n_sims = 10000 # number of simulation runs
n0_data = 500 # sample size logged from policy 0 (algorithm A)
n1_data = 500 # sample size logged from policy 1 (algorithm B)
n_actions = 30 # number of actions
reward_type = "continuous" # binary or continuous

# policy params
# - policy_kind: the kind of conditioned action distribution
# - mu_rate: the peak position rate of the action distribution
# - sigma: the length of tail
policy_0_params = {"policy_kind": "sorted_normal", "mu_rate": 0.5, "sigma": 8.0}
policy_1_params = {"policy_kind": "sorted_normal", "mu_rate": 0.0, "sigma": 8.0}
reward_std_list = [
    0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5,
    5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0
] # reward noise levels to vary

In [3]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [ ]:
def estimate(reward_std):
    dataset = KuaiRecDataset(
        n_actions=n_actions,
        reward_std=reward_std,
        random_state=os.getpid()
    )

    true_value_of_pi_0 = dataset.calc_policy_value(policy_params=policy_0_params)
    true_value_of_pi_1 = dataset.calc_policy_value(policy_params=policy_1_params)
    true_value_of_pi_m = dataset.calc_mid_policy_value(
        policy_0_params=policy_0_params, policy_1_params=policy_1_params
    )
    is_pi_1_better = np.int32(true_value_of_pi_0 < true_value_of_pi_1)
    is_pi_1_better = is_pi_1_better - (1 - is_pi_1_better)

    estimated_pi_0_value_list = []
    estimated_pi_1_value_list = []
    estimated_pi_m_D_1_value_list = []
    estimated_pi_m_D_0_value_list = []
    estimated_comparison_list = []
    estimated_p_value_list = []

    for _ in range(n_sims):
        D_E_0 = dataset.generate_dataset(
            n_data=n0_data,
            logging_policy_params=policy_0_params,
            evaluation_policy_params=policy_1_params
        )
        D_E_1 = dataset.generate_dataset(
            n_data=n1_data,
            logging_policy_params=policy_1_params,
            evaluation_policy_params=policy_0_params
        )

        values_of_pi_1, values_of_pi_m_D_1, values_of_pi_m_D_0, values_of_pi_0, comparison, p_value = run_experiment([D_E_0, D_E_1])
        estimated_pi_1_value_list.append(values_of_pi_1)
        estimated_pi_m_D_1_value_list.append(values_of_pi_m_D_1)
        estimated_pi_m_D_0_value_list.append(values_of_pi_m_D_0)
        estimated_pi_0_value_list.append(values_of_pi_0)
        estimated_comparison_list.append(comparison)
        estimated_p_value_list.append(p_value)

    result_df = (
        pd.concat([
            pd.DataFrame(estimated_pi_1_value_list).stack(),
            pd.DataFrame(estimated_pi_m_D_1_value_list).stack(),
            pd.DataFrame(estimated_pi_m_D_0_value_list).stack(),
            pd.DataFrame(estimated_pi_0_value_list).stack(),
            pd.DataFrame(estimated_comparison_list).stack(),
            pd.DataFrame(estimated_p_value_list).stack(),
        ], axis=1)
        .reset_index(1)
        .rename(columns={
            "level_1": "est",
            0: "pi_1_value", 1: "pi_m_D_1_value", 2: "pi_m_D_0_value",
            3: "pi_0_value", 4: "selection", 5: "p_value"
        })
    )
    result_df["reward_std"] = reward_std
    result_df["true_pi_0_value"] = true_value_of_pi_0
    result_df["true_pi_1_value"] = true_value_of_pi_1
    result_df["true_pi_m_value"] = true_value_of_pi_m
    result_df["selection"] *= is_pi_1_better
    result_df["selection"] = result_df["selection"] == 1

    return result_df

In [5]:
with tqdm_joblib(tqdm(total=len(reward_std_list))):
    result_df_list = Parallel(n_jobs=-1, verbose=0)(
        [delayed(estimate)(reward_std) for reward_std in reward_std_list]
    )

result_df = pd.concat(result_df_list).reset_index(level=0)
result_df

100%|██████████| 21/21 [02:56<00:00,  8.41s/it]


,index,est,pi_1_value,pi_m_D_1_value,pi_m_D_0_value,pi_0_value,selection,p_value,reward_std,true_pi_0_value,true_pi_1_value,true_pi_m_value
0,0,AVG,0.556327,0.000000,0.000000,1.012595,True,1.244240e-27,0.0,0.977944,0.540052,0.442872
1,0,IPS,0.545131,0.000000,0.000000,1.012595,True,4.640963e-22,0.0,0.977944,0.540052,0.442872
2,0,AVG_IPS,0.556327,0.782294,0.778863,1.012595,True,1.322404e-07,0.0,0.977944,0.540052,0.442872
3,0,MIN,0.556327,0.342153,0.340166,1.012595,True,9.211322e-25,0.0,0.977944,0.540052,0.442872
4,0,MID,0.556327,0.475513,0.461377,1.012595,True,1.262075e-23,0.0,0.977944,0.540052,0.442872
...,...,...,...,...,...,...,...,...,...,...,...,...
1049995,9999,AVG,0.267743,0.000000,0.000000,1.085379,True,2.040819e-01,10.0,0.977944,0.540052,0.442872
1049996,9999,IPS,0.060401,0.000000,0.000000,1.085379,True,3.559144e-01,10.0,0.977944,0.540052,0.442872
1049997,9999,AVG_IPS,0.267743,0.948460,0.572890,1.085379,True,9.156605e-02,10.0,0.977944,0.540052,0.442872
1049998,9999,MIN,0.267743,0.532142,0.564865,1.085379,True,5.694402e-02,10.0,0.977944,0.540052,0.442872


## Analyze

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
plt.rcParams["font.size"] = 12
for est_name in ["AVG", "IPS", "MID"]:
    mask_est = result_df["est"] == est_name
    result_dict = {
        "error rate": [],
        "error rate upper": [],
        "error rate lower": [],
        "error rate interval": [],
        "squared_bias": [],
        "variance": [],
        "MSE": [],
    }
    
    for reward_std in reward_std_list:
        mask_reward_std = result_df["reward_std"] == reward_std
        mask = mask_est & mask_reward_std
        result = result_df[mask].copy()
        performance = evaluate_performance(result)
        
        result["delta_value"] = result["pi_1_value"] - result["pi_m_D_1_value"] + result["pi_m_D_0_value"] - result["pi_0_value"]
        est_selection = result["delta_value"] > 0
        m, e, d = np.mean(est_selection), stats.sem(est_selection), len(est_selection) - 1
        interval = stats.t.interval(0.95, d, loc=m, scale=e)
        
        result_dict["error rate"].append(1.0-performance["accuracy"])
        result_dict["error rate lower"].append(1.0-interval[0])
        result_dict["error rate upper"].append(1.0-interval[1])
        result_dict["error rate interval"].append(interval[1]-interval[0])
        result_dict["squared_bias"].append(performance["squared_bias"])
        result_dict["variance"].append(performance["variance"])
        result_dict["MSE"].append(performance["MSE"])
    
    axes[0].errorbar(reward_std_list, result_dict["error rate"], yerr=result_dict["error rate interval"], label=f"{est_name}")
    axes[0].set_ylim(0.0, 0.60)
    axes[0].set_xlabel("reward noise")
    axes[0].set_title("error rate")
    
    axes[1].plot(reward_std_list, result_dict["variance"], label=f"{est_name}")
    axes[1].set_xlabel("reward noise")
    axes[1].set_title("variance")
    axes[1].set_ylim(0, 1.1)
    
    mask_reward_std = result_df["reward_std"] == 5.0
    mask = mask_est & mask_reward_std
    result = result_df[mask].copy()
    
    alpha_grid = np.linspace(0.001, 0.25, 60)
    power = np.array([(result["p_value"] < alpha).mean() for alpha in alpha_grid])
    axes[2].plot(alpha_grid, power, label=f'{est_name}', lw=2)
    axes[2].set_title(f"power (reward noise = {reward_std})")
    axes[2].set_xlabel("level of significance")
    axes[2].set_ylim(0, 1.1)

plt.tight_layout()
axes[1].legend(["AVG (A/B test)", "IPS (offline)", "MID(Ours; A/B test)"], loc="upper center", bbox_to_anchor=(0.5, 1.40), ncol=3, fontsize=15)
plt.show()